# 06 — Delta Lake time travel viewer

Interactive surface to **re-read historical snapshots** of the local Delta tables. Delta Lake keeps a transaction log, so every commit is an addressable `version`. No dbt, no extra service — this is native Delta time travel exposed through `src/`.

**Questions answered:** Which versions exist for each Bronze/Silver table? What did a table look like at version *N*? What changed between two versions (row count, schema)?

**Two kinds of “time” — do not confuse them.** Delta `version` is *physical/audit* history (which commit produced the files). It is **not** the PIT feature cutoff, which is enforced by `event_timestamp` / `knowledge_step` in the feature engines. Pinning an old Delta version is for reproducibility and inspection, never a substitute for cutoff logic.

This notebook stays **review-only / output-free** in git (AGENTS §13): it calls into `src/`, runs read-only, and must be cleared before commit.

In [1]:
from pathlib import Path

import pandas as pd
from deltalake import DeltaTable
from IPython.display import Markdown, display

from pit_fintech.config import get_settings
from pit_fintech.data.paysim import resolve_project_root
from pit_fintech.data.paysim_lakehouse import (
    ApplicationLakehouseManifest,
    _resolve_manifest_path,
    find_latest_paysim_lakehouse_manifest,
    paysim_lakehouse_history,
)

# Which application lakehouse to inspect. The PaySim Bronze/Silver tables are the real data.
DATASET = "paysim"

PROJECT_ROOT = resolve_project_root(Path.cwd())
artifact_root = get_settings().artifact_root
if not artifact_root.is_absolute():
    artifact_root = PROJECT_ROOT / artifact_root

manifest_path = find_latest_paysim_lakehouse_manifest(artifact_root)
MANIFEST_AVAILABLE = manifest_path is not None

if not MANIFEST_AVAILABLE:
    display(
        Markdown(
            "> **No PaySim lakehouse manifest found.** Run "
            "`.\\make.ps1 build-lakehouse -Dataset paysim` first, then reload this notebook."
        )
    )
else:
    print(f"manifest: {manifest_path}")

manifest: C:\workspace\pit-fintech\artifacts\datasets\paysim1\16910f90577b0d98\lakehouse\lakehouse-manifest.json


## 1 · Which versions exist?

Every published Bronze/Silver table and its full Delta commit history — the `version` column is what you pin below. This mirrors `pit data lakehouse-history --dataset paysim` but renders as a DataFrame for easier scanning.

In [2]:
if MANIFEST_AVAILABLE:
    history = paysim_lakehouse_history(manifest_path, project_root=PROJECT_ROOT)
    history_df = pd.DataFrame(history)
    wanted = ["layer", "table", "version", "operation", "timestamp"]
    keep = [c for c in wanted if c in history_df.columns]
    history_df = history_df[keep].sort_values(["table", "version"]).reset_index(drop=True)
    display(history_df)
else:
    print("Skipped: build the PaySim lakehouse first.")

,layer,table,version,operation,timestamp
0,silver,paysim_labels,0,WRITE,1785135531449
1,silver,paysim_labels,1,WRITE,1785138379892
2,silver,paysim_labels,2,WRITE,1785139886175
3,silver,paysim_labels,3,WRITE,1785310299210
4,silver,paysim_labels,4,WRITE,1785310898404
5,silver,paysim_labels,5,WRITE,1785314647792
6,silver,paysim_labels,6,WRITE,1785315321503
7,silver,paysim_labels,7,WRITE,1785740044878
8,bronze,paysim_transactions,0,WRITE,1785135500681
9,silver,paysim_transactions,0,WRITE,1785135519110


## 2 · Resolve table paths

Build a `{table: path}` map from the manifest using the same resolver the CLI uses, so the paths are exactly what the pipeline wrote.

In [3]:
TABLES: dict[str, dict] = {}
if MANIFEST_AVAILABLE:
    manifest = ApplicationLakehouseManifest.model_validate_json(
        manifest_path.read_text(encoding="utf-8")
    )
    for snap in manifest.tables:
        TABLES[snap.table] = {
            "layer": snap.layer,
            "path": _resolve_manifest_path(snap.path, PROJECT_ROOT),
            "published_version": snap.version,
        }
    display(
        pd.DataFrame(
            [
                {
                    "table": t,
                    "layer": v["layer"],
                    "published_version": v["published_version"],
                    "path": str(v["path"]),
                }
                for t, v in TABLES.items()
            ]
        )
    )
else:
    print("Skipped: no manifest.")

,table,layer,published_version,path
0,paysim_transactions,silver,7,C:\workspace\pit-fintech\data\lakehouse\paysim...
1,paysim_labels,silver,7,C:\workspace\pit-fintech\data\lakehouse\paysim...


## 3 · Read one snapshot at a specific version

Set `TABLE` and `VERSION` from the history table above, then preview the rows exactly as they were at that commit. This is `DeltaTable(path, version=N).to_pyarrow_table()` — the same call `pit model gold-evaluate` uses to pin exact versions.

In [18]:
# --- Parameters: edit these ---
TABLE = "paysim_transactions"  # a key from TABLES above (silver txns, bronze txns, labels, ...)
VERSION = 7  # a version from the history table; None = latest
PREVIEW_ROWS = 10
# ------------------------------


def read_version(table: str, version: int | None):
    """Read a Delta table at an exact version (None = latest) as a pyarrow Table."""
    if table not in TABLES:
        raise KeyError(f"Unknown table {table!r}. Choose from: {sorted(TABLES)}")
    path = str(TABLES[table]["path"])
    dt = DeltaTable(path) if version is None else DeltaTable(path, version=version)
    return dt.to_pyarrow_table(), dt.version()


if TABLES:
    snapshot, resolved_version = read_version(TABLE, VERSION)
    display(
        Markdown(
            f"**{TABLE}** @ version **{resolved_version}** — "
            f"{snapshot.num_rows:,} rows, {snapshot.num_columns} columns"
        )
    )
    display(snapshot.slice(0, PREVIEW_ROWS).to_pandas())
else:
    print("Skipped: no tables resolved.")

**paysim_transactions** @ version **7** — 6,362,620 rows, 13 columns

,source_row_number,step,knowledge_step,transaction_type,amount,origin_entity_id,origin_entity_kind,destination_entity_id,destination_entity_kind,dataset_snapshot_id,source_file_sha256,source_record_id,event_day
0,5987418,409,409,PAYMENT,3502.50,C668582411,CUSTOMER,M581144276,MERCHANT,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987418,18
1,5987419,409,409,PAYMENT,13292.15,C152122242,CUSTOMER,M783360034,MERCHANT,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987419,18
2,5987420,409,409,PAYMENT,4516.12,C1200715365,CUSTOMER,M2038076781,MERCHANT,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987420,18
3,5987421,409,409,PAYMENT,3318.67,C2046239971,CUSTOMER,M1591786447,MERCHANT,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987421,18
4,5987422,409,409,TRANSFER,57167.58,C311348264,CUSTOMER,C1477062066,CUSTOMER,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987422,18
5,5987423,409,409,CASH_OUT,136398.48,C1064493801,CUSTOMER,C882644953,CUSTOMER,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987423,18
6,5987424,409,409,CASH_IN,138560.64,C2145520088,CUSTOMER,C413957103,CUSTOMER,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987424,18
7,5987425,409,409,CASH_IN,437540.21,C612891432,CUSTOMER,C220882828,CUSTOMER,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987425,18
8,5987426,409,409,CASH_IN,254004.73,C1074568220,CUSTOMER,C2110835918,CUSTOMER,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987426,18
9,5987427,409,409,CASH_IN,159110.69,C1757502302,CUSTOMER,C1750003248,CUSTOMER,paysim1:16910f90577b0d98,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f...,paysim1:16910f90577b0d98:5987427,18


## 4 · Compare two versions

Diff a table between two commits: row-count delta and any schema (column) change. Useful to see what a rebuild (e.g. Silver v4 → v6 in M026/M027) actually moved.

In [13]:
# --- Parameters: edit these ---
DIFF_TABLE = "paysim_transactions"
VERSION_A = 0
VERSION_B = None  # None = latest
# ------------------------------

if TABLES:
    a, va = read_version(DIFF_TABLE, VERSION_A)
    b, vb = read_version(DIFF_TABLE, VERSION_B)
    cols_a, cols_b = set(a.column_names), set(b.column_names)
    summary = pd.DataFrame(
        [
            {"metric": "version", f"v{va}": va, f"v{vb}": vb},
            {"metric": "rows", f"v{va}": a.num_rows, f"v{vb}": b.num_rows},
            {"metric": "columns", f"v{va}": a.num_columns, f"v{vb}": b.num_columns},
        ]
    )
    display(summary)
    display(
        Markdown(
            f"- rows delta: **{b.num_rows - a.num_rows:+,}**\n"
            f"- columns only in v{va}: `{sorted(cols_a - cols_b) or 'none'}`\n"
            f"- columns only in v{vb}: `{sorted(cols_b - cols_a) or 'none'}`"
        )
    )
else:
    print("Skipped: no tables resolved.")

,metric,v0,v7
0,version,0,7
1,rows,6362620,6362620
2,columns,12,13


- rows delta: **+0**
- columns only in v0: `none`
- columns only in v7: `['knowledge_step']`

## 5 · Read any Delta path directly (optional)

Gold tables live outside the Bronze/Silver manifest. Point `ANY_PATH` at any Delta directory (e.g. `data/lakehouse/paysim1/16910f90577b0d98/gold/pre_decision_features`) to time-travel it too.

In [6]:
# --- Parameters: edit these ---
ANY_PATH = ""  # e.g. "data/lakehouse/paysim1/16910f90577b0d98/gold/pre_decision_features"
ANY_VERSION = None  # None = latest
# ------------------------------

if ANY_PATH:
    p = Path(ANY_PATH)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    dt = DeltaTable(str(p)) if ANY_VERSION is None else DeltaTable(str(p), version=ANY_VERSION)
    display(pd.DataFrame(dt.history()))
    tbl = dt.to_pyarrow_table()
    display(
        Markdown(
            f"**{p.name}** @ version **{dt.version()}** — "
            f"{tbl.num_rows:,} rows, {tbl.num_columns} columns"
        )
    )
    display(tbl.slice(0, 10).to_pandas())
else:
    print("Set ANY_PATH to inspect a Delta table outside the Bronze/Silver manifest (e.g. Gold).")

Set ANY_PATH to inspect a Delta table outside the Bronze/Silver manifest (e.g. Gold).


---
**Reminder.** Clear all outputs before committing (`review-only`). Delta `version` pinning is physical/audit reproducibility; PIT feature correctness is enforced separately by `event_timestamp` / `knowledge_step`. Never use version pinning as a cutoff.